In [5]:
from transformers import EsmTokenizer, EsmForMaskedLM

model_name = 'facebook/esm1v_t33_650M_UR90S_1'
tokenizer = EsmTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

Some weights of EsmForMaskedLM were not initialized from the model checkpoint at facebook/esm1v_t33_650M_UR90S_1 and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
from datasets import load_from_disk

tokenized_dataset = load_from_disk('../../data/training/maskedlm/fnbp-70K-tokenized')

In [7]:
# Create a subset temporarily
tokenized_dataset = tokenized_dataset['train'].train_test_split(train_size=90, test_size=10, shuffle=True)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 90
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10
    })
})

In [8]:
import torch

def compute_metrics(eval_pred):
    # Convert Hugging Face outputs to PyTorch tensors
    logits, labels = eval_pred
    logits = torch.tensor(logits)  # Shape: (batch_size, seq_len, vocab_size=33)
    labels = torch.tensor(labels)  # Shape: (batch_size, seq_len)
    
    # 1. Identify masked tokens (where labels != -100)
    mask = labels != -100  # Boolean mask for prediction positions
    num_masked_tokens = mask.sum().item()  # Count masked tokens
    
    # Handle edge case: no masked tokens
    if num_masked_tokens == 0:
        return {'pppl_global': float('nan')}

    # 2. Compute log probabilities
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)  # Shape: (batch_size, seq_len, vocab_size)
    
    # 3. Create safe labels: replace -100 with 0 (to avoid out-of-bounds errors)
    safe_labels = torch.where(labels == -100, 0, labels).long()
    
    # 4. Gather true token log probabilities
    token_log_probs = torch.gather(  # Gather along vocabulary dimension
        input=log_probs, 
        dim=-1, 
        index=safe_labels.unsqueeze(-1)  # Add extra dim for gathering
    ).squeeze(-1)  # Shape: (batch_size, seq_len)
    
    # 5. Compute NLL only for masked tokens
    nll = -token_log_probs[mask]  # Shape: (num_masked_tokens,)
    
    # 6. Calculate pseudo-perplexity
    avg_nll = nll.mean()  # Average NLL over masked tokens
    pppl_global = torch.exp(avg_nll).item()  # exp(avg_nll)
    
    return {'pppl_global': pppl_global}

In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

mlm_ouptut_dir = '../../outputs/mlm'
model_id = 'SpySeer-650M-FNBP-70K'

training_args = TrainingArguments(
    output_dir=f'{mlm_ouptut_dir}/{model_id}',
    num_train_epochs=2,
    save_strategy='epoch',
    save_total_limit=5,
    logging_strategy='steps',
    logging_steps=10,
    eval_strategy='epoch',
    learning_rate=4e-4,
    weight_decay=0.01,
    per_device_eval_batch_size=1,
    per_device_train_batch_size=1,
    push_to_hub=True,
    hub_model_id=model_id,
    report_to='tensorboard',
    load_best_model_at_end=True,
    metric_for_best_model='eval_pppl_global'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [14]:
trainer.state.best_model_checkpoint

'../../outputs/mlm/SpySeer-8M\\checkpoint-46'

In [17]:
best_checkpoint = trainer.state.best_model_checkpoint
model = EsmForMaskedLM.from_pretrained(best_checkpoint)

In [18]:
model

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate